In [ ]:
import json
import pandas
from collections import defaultdict

In [ ]:
with open("../../data/raw/rtl_comments_clean_2012-2024.json") as f:
    comments = json.load(f)

len(comments)

In [ ]:
liked_comments = []
for comment in comments:
    if comment["thumbs"] is not None:
        liked_comments.append(comment)

len(liked_comments)

In [ ]:
comment_interactions_per_user = defaultdict(list)

for comment in liked_comments:
    for user in comment["thumbs"]:
        comment_interactions_per_user[user["user_id"]].append(comment)

In [ ]:
users = []

for key, comments in comment_interactions_per_user.items():
    cleaned_interactions = {"likes": [], "dislikes": []}

    for comment in comments:
        for user in comment["thumbs"]:
            # drop all comments that start with an @
            if comment["text"].strip().startswith("@"):
                continue
            if user["user_id"] == key:
                cleaned_comment = {
                    "id": comment["context_id"],
                    "text": comment["text"],
                    "date": comment["date_created"],
                    "user_id": comment["user_id"],
                    "lang": comment["lang_id"],
                }
                if user["score"] == "up":
                    cleaned_interactions["likes"].append(cleaned_comment)
                    
                elif user["score"] == "down":
                    cleaned_interactions["dislikes"].append(cleaned_comment)
    
    # deduplicate interactions
    cleaned_interactions["likes"] = list({interaction["id"]: interaction for interaction in cleaned_interactions["likes"]}.values())
    cleaned_interactions["dislikes"] = list({interaction["id"]: interaction for interaction in cleaned_interactions["dislikes"]}.values())


    # user comments
    user_comments = []
    for comment in comments:
        if comment["user_id"] == key:
            cleaned_comment = {
                "id": comment["context_id"],
                "text": comment["text"],
                "date": comment["date_created"],
                "user_id": comment["user_id"],
                "lang": comment["lang_id"],
            }
            user_comments.append(cleaned_comment)

    # deduplicate user comments
    user_comments = list({comment["id"]: comment for comment in user_comments}.values())

    # drop self interactions
    cleaned_interactions["likes"] = [interaction for interaction in cleaned_interactions["likes"] if interaction["user_id"] != key]
    cleaned_interactions["dislikes"] = [interaction for interaction in cleaned_interactions["dislikes"] if interaction["user_id"] != key]

    users.append({
        "user_id": key,
        "comment_count": len(user_comments),
        "like_count": len(cleaned_interactions["likes"]),
        "dislike_count": len(cleaned_interactions["dislikes"]),
        "total_interactions": len(cleaned_interactions["likes"]) + len(cleaned_interactions["dislikes"]),
        "like_ratio": len(cleaned_interactions["likes"]) / (len(cleaned_interactions["likes"]) + len(cleaned_interactions["dislikes"])) if (len(cleaned_interactions["likes"]) + len(cleaned_interactions["dislikes"])) > 0 else 0,
        "interactions": cleaned_interactions,
        "comments": user_comments
    })

# deduplicate users
users = list({user["user_id"]: user for user in users}.values())

with open("../../data/processed/cleaned_users.json", "w") as f:
    json.dump(users, f, indent=4)